# Pipeline 5 - Water Fraction threshold

In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
OUTPUT_DIR = "/data/swot/output"

## Read the sites and dates

In [18]:
import numpy as np
import pandas as pd

from swot_toolkit.analysis import open_sites_and_dates
from swot_toolkit.metrics import calc_metrics, process_swot_mask
from swot_toolkit.pipe4 import create_swot_mosaic, open_datasets, quality_flags_bad

sites_dates = open_sites_and_dates("/data/swot/output")

METRICS = ["iou", "f1", "precision", "recall"]

In [19]:
metrics = ["iou", "f1", "precision", "recall"]

overall_results = pd.DataFrame()
for site, dates in sites_dates.items():
    for date in dates:
        # Load the datasets
        datasets = open_datasets(site, date)

        # Now, let's create the Raster 100 array
        swot_mask, _, _ = create_swot_mosaic(
            region_name=site,
            ref_date=date,
            exclude_flags=quality_flags_bad,
            exclude_no_data=True,
        )

        # Once the datasets are open, we can apply the threshold
        results = {}
        for thresh in np.arange(0, 1.0, 0.01):
            mask = process_swot_mask(swot_mask, water_threshold=thresh)

            title = f"water >= {thresh:.2f}"
            results[title] = calc_metrics(
                datasets["ref_mask"],
                mask,
                metrics,
                binary=True,
            ).rename(columns={0: title})

        metrics_df = pd.concat(results.values(), axis=1)
        metrics_df.index = pd.MultiIndex.from_product([[f"{site} {date}"], metrics_df.index])
        overall_results = pd.concat([overall_results, metrics_df], axis=0)


No OPERA S1 mask available for date 202407
The following datasets have been opened: ['s2_img', 'scl', 'ref_mask', 'opera_s2']
The following datasets have been opened: ['s2_img', 'scl', 'ref_mask', 'opera_s2', 'opera_s1']
No OPERA S1 mask available for date 202405
The following datasets have been opened: ['s2_img', 'scl', 'ref_mask', 'opera_s2']
The following datasets have been opened: ['s2_img', 'scl', 'ref_mask', 'opera_s2', 'opera_s1']
No OPERA S1 mask available for date 202403
The following datasets have been opened: ['s2_img', 'scl', 'ref_mask', 'opera_s2']
The following datasets have been opened: ['s2_img', 'scl', 'ref_mask', 'opera_s2', 'opera_s1']
No OPERA S1 mask available for date 202408
The following datasets have been opened: ['s2_img', 'scl', 'ref_mask', 'opera_s2']
The following datasets have been opened: ['s2_img', 'scl', 'ref_mask', 'opera_s2', 'opera_s1']
No OPERA S1 mask available for date 202411
The following datasets have been opened: ['s2_img', 'scl', 'ref_mask', 'o

In [5]:
site_means = overall_results.groupby(level=1).mean()
site_means.index = pd.MultiIndex.from_product([["MEAN"], site_means.index])

site_medians = overall_results.groupby(level=1).median()
site_medians.index = pd.MultiIndex.from_product([["MEDIAN"], site_medians.index])

overall_results = pd.concat([overall_results, site_means, site_medians], axis=0)


In [15]:
columns = site_means.columns[:]
overall_results[columns]

water >= 0.00  water >= 0.05  water >= 0.10  \
Curua-Una 2024-07-13   iou              0.02470        0.42520        0.48930   
                       f1               0.04820        0.59670        0.65710   
                       precision        0.02470        0.43130        0.49970   
                       recall           0.99950        0.96760        0.95930   
Curua-Una 2025-08-14   iou              0.02890        0.56240        0.64160   
                       f1               0.05610        0.71990        0.78170   
                       precision        0.02890        0.57410        0.66200   
                       recall           0.99990        0.96500        0.95420   
Northeast 2024-05-29   iou              0.02110        0.35170        0.45760   
                       f1               0.04140        0.52040        0.62780   
                       precision        0.02110        0.35700        0.47160   
                       recall           0.99940        0.95950        0.93890   
Northeast 2025-07-20   iou              0.01790        0.33380        0.43940   
                       f1               0.03520        0.50060        0.61060   
                       precision        0.01790        0.33460        0.44140   
                       recall           0.99980        0.99290        0.98990   
Rio_Branco 2024-04-03  iou              0.02950        0.22360        0.24340   
                       f1               0.05740        0.36550        0.39150   
                       precision        0.02950        0.23900        0.26630   
                       recall           0.99520        0.77640        0.73900   
Rio_Branco 2025-09-07  iou              0.05560        0.36290        0.38410   
                       f1               0.10530        0.53250        0.55500   
                       precision        0.05560        0.38390        0.41340   
                       recall           0.99520        0.86920        0.84410   
Rio_Madeira 2024-08-21 iou              0.05360        0.56870        0.60740   
                       f1               0.10170        0.72500        0.75570   
                       precision        0.05360        0.57400        0.61580   
                       recall           0.99990        0.98380        0.97780   
Rio_Madeira 2025-07-21 iou              0.05980        0.50310        0.55260   
                       f1               0.11280        0.66940        0.71190   
                       precision        0.05980        0.50820        0.56120   
                       recall           0.99960        0.98050        0.97310   
Rio_Negro 2024-11-29   iou              0.16220        0.46000        0.48180   
                       f1               0.27910        0.63010        0.65030   
                       precision        0.16220        0.46100        0.48360   
                       recall           0.99990        0.99520        0.99210   
Rio_Negro 2025-08-07   iou              0.26200        0.68510        0.70970   
                       f1               0.41520        0.81310        0.83020   
                       precision        0.26200        0.69240        0.72110   
                       recall           0.99940        0.98480        0.97830   
MEAN                   f1               0.12524        0.60732        0.65718   
                       iou              0.07153        0.44765        0.50069   
                       precision        0.07153        0.45555        0.51361   
                       recall           0.99878        0.94749        0.93467   
MEDIAN                 f1               0.07955        0.61340        0.65370   
                       iou              0.04155        0.44260        0.48555   
                       precision        0.04155        0.44615        0.49165   
                       recall           0.99955        0.97405        0.96620   

                                  water >= 0.15  water >= 0.20  

In [7]:
import plotly.graph_objects as go


In [8]:
styles = {
    "precision": {"line": {"width": 1, "color": "blue", "dash": "dash"}},
    "recall": {"line": {"width": 1, "color": "green", "dash": "dash"}},
    "f1": {"line": {"width": 3, "color": "purple", "dash": "solid"}},
    "iou": {"line": {"width": 3, "color": "red", "dash": "solid"}},
}

In [9]:
site_means.index.get_level_values(1)

Index(['f1', 'iou', 'precision', 'recall'], dtype='object')

### MEAN graph

In [10]:
fig = go.Figure()

for idx, row in site_means.iterrows():
    x = [float(c[-4:]) for c in site_means.columns]
    name = idx[1] if isinstance(idx, tuple) else idx
    scatter = go.Scatter(x=x, y=row, mode="lines", name=name, line=styles[name]["line"])
    fig.add_trace(scatter)

# Add axis labels
fig.update_layout(xaxis_title="Water Fraction Threshold", yaxis_title="Values")


loc = ("MEAN", "f1") if isinstance(idx, tuple) else "f1"
# Find F1 value at x=0.55 and add annotation
f1_at_055 = site_means.loc[loc, "water >= 0.60"]
fig.add_annotation(
    x=0.55,
    y=f1_at_055,
    text=f"max(F1): x=0.55, y={f1_at_055:.3f}",
    showarrow=True,
    arrowhead=2,
    arrowsize=1,
    arrowwidth=2,
    arrowcolor="purple",
    bgcolor="white",
    bordercolor="purple",
    borderwidth=1,
)

fig

### MEDIAN graph

In [13]:
site_medians[site_medians.columns[::2]]

water >= 0.00  water >= 0.10  water >= 0.20  water >= 0.30  \
MEDIAN f1               0.07955        0.65370        0.73225        0.77460   
       iou              0.04155        0.48555        0.57755        0.63215   
       precision        0.04155        0.49165        0.60005        0.66865   
       recall           0.99955        0.96620        0.94975        0.93280   

                  water >= 0.40  water >= 0.50  water >= 0.60  water >= 0.70  \
MEDIAN f1               0.79180        0.81230        0.86220        0.84965   
       iou              0.65535        0.68430        0.75785        0.73865   
       precision        0.70800        0.75335        0.88570        0.90770   
       recall           0.90955        0.87440        0.83450        0.78915   

                  water >= 0.80  water >= 0.90  
MEDIAN f1               0.82190        0.76250  
       iou              0.69765        0.61615  
       precision        0.92060        0.93175  
       recall           0.74000        0.64625

In [14]:
fig = go.Figure()

for idx, row in site_medians.iterrows():
    x = [float(c[-4:]) for c in site_medians.columns]
    name = idx[1] if isinstance(idx, tuple) else idx
    scatter = go.Scatter(x=x, y=row, mode="lines", name=name, line=styles[name]["line"])
    fig.add_trace(scatter)

# Add axis labels
fig.update_layout(xaxis_title="Water Fraction Threshold", yaxis_title="Values")


loc = ("MEDIAN", "f1") if isinstance(idx, tuple) else "f1"
# Find max F1 value and corresponding x to add annotation
y_max = site_medians.iloc[0].max()
x_y_max = site_medians.iloc[0].idxmax().split(" >= ")[-1]

fig.add_annotation(
    x=x_y_max,
    y=y_max,
    text=f"max(F1): x={x_y_max}, y={y_max:.3f}",
    showarrow=True,
    arrowhead=2,
    arrowsize=1,
    arrowwidth=2,
    arrowcolor="purple",
    bgcolor="white",
    bordercolor="purple",
    borderwidth=1,
)

fig.update_layout(title="Median Metrics across Sites")

### Only IoU

In [ ]:
site_medians[site_medians.columns[::2]]

water >= 0.00  water >= 0.10  water >= 0.20  water >= 0.30  \
MEDIAN f1                0.1053         0.7119         0.7502         0.8005   
       iou               0.0556         0.5526         0.6002         0.6674   
       precision         0.0556         0.5612         0.6145         0.6873   
       recall            0.9996         0.9731         0.9628         0.9488   

                  water >= 0.40  water >= 0.50  water >= 0.60  water >= 0.70  \
MEDIAN f1                0.8420         0.8613         0.8738         0.8551   
       iou               0.7271         0.7564         0.7759         0.7469   
       precision         0.7706         0.8348         0.8979         0.9145   
       recall            0.9280         0.8896         0.8398         0.7841   

                  water >= 0.80  water >= 0.90  
MEDIAN f1                0.8240         0.7646  
       iou               0.7007         0.6189  
       precision         0.9286         0.9392  
       recall            0.7394         0.6387

In [64]:
fig = go.Figure()

for idx, row in site_medians.iterrows():
    name = idx[1] if isinstance(idx, tuple) else idx

    if name != "iou":
        continue

    x = [float(c[-4:]) for c in site_medians.columns]
    scatter = go.Scatter(x=x, y=row, mode="lines", name=name, line=styles[name]["line"])
    fig.add_trace(scatter)

# Add axis labels
axis_style = {
    "dtick": 0.1,
    "range": [0.05, .95],
    "showline": True,
    "linecolor": "black",
    "mirror": True,
    "showgrid": True,
    "gridcolor": "lightgrey",
    "linewidth": 2,
}

yaxis = axis_style.copy()
yaxis.update({"title": "IoU Score"})
xaxis = axis_style.copy()
xaxis.update({"title": "Water Fraction Threshold"})

fig.update_layout(
    title="Median IoU across Sites",
    plot_bgcolor="white",
    yaxis=yaxis,
    xaxis=xaxis,
)


In [59]:
xaxis

In [49]:
dir(d)

['__class__',
 '__class_getitem__',
 '__contains__',
 '__delattr__',
 '__delitem__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__ior__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__ne__',
 '__new__',
 '__or__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__reversed__',
 '__ror__',
 '__setattr__',
 '__setitem__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'clear',
 'copy',
 'fromkeys',
 'get',
 'items',
 'keys',
 'pop',
 'popitem',
 'setdefault',
 'update',
 'values']

In [51]:
d.update({3: 4})

In [52]:
d

{2: 1, 3: 4}